# Effective rank summary

In [ ]:
[effrank]
parameter: res_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/res")
parameter: pattern = "res_*.rds"
parameter: cwd = path("output")
# ---- coalescent expectation knobs -------------------------------------------
parameter: Ne = 10000          # effective population size (European)
parameter: C = 6               # coalescent constant C in [5,7]
parameter: cm_per_mb = 1.0     # rough cM per Mb (swap for a real map factor)
# ---- job knobs --------------------------------------------------------------
parameter: numThreads = 3
parameter: walltime = "24:00"
parameter: mem = "16G"
parameter: container = ""

import glob, os
res_files = sorted(glob.glob(os.path.join(f"{res_dir}", pattern)))
if len(res_files) == 0:
    raise ValueError(f"No files matched {os.path.join(str(res_dir), pattern)}")

input: res_files
output: f"{cwd:a}/effective_rank_summary.rds", f"{cwd:a}/effective_rank_summary.csv"
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container

    suppressMessages(library(parallel))
    files     <- c(${_input:r,})
    Ne        <- ${Ne}
    C         <- ${C}
    cm_per_mb <- ${cm_per_mb}
    `%||%` <- function(a, b) if (is.null(a) || length(a) == 0) b else a

    process_one <- function(f) tryCatch({
      res  <- readRDS(f)
      gene <- sub("^res_(.*)\\.rds$", "\\1", basename(f))
      if (length(res) < 1) return(NULL)

      rr <- res[[1]]$susie_eb_mix_res          # all elements are the same region -> [[1]]
      if (is.null(rr)) return(data.frame(
        gene = gene, n_variants = NA_integer_, span_Mb = NA_real_, rho_cM = NA_real_,
        eff_rank_emp = NA_real_, eff_rank_exp = NA_real_, ratio_emp_exp = NA_real_,
        B_rec_emp = NA_real_, row.names = NULL))

      ## empirical effective rank from the finite-reference diagnostics
      eff_emp <- as.numeric(rr$R_finite_diagnostics$effective_rank %||% NA_real_)

      ## expected effective rank from the variant span:  C * Ne * rho
      ids <- names(rr$pip)
      if (length(ids) == 0) {
        p <- NA_integer_; span_Mb <- NA_real_; rho_cM <- NA_real_; eff_exp <- NA_real_
      } else {
        pos     <- as.integer(sub("^chr[^:]*:([0-9]+):.*$", "\\1", ids))
        p       <- length(ids)
        span_Mb <- (max(pos) - min(pos)) / 1e6
        rho_cM  <- span_Mb * cm_per_mb          # ROUGH: ~1 cM/Mb until a real map is used
        eff_exp <- C * Ne * (rho_cM / 100)      # rho in Morgans
      }

      data.frame(
        gene          = gene,
        n_variants    = p,
        span_Mb       = span_Mb,
        rho_cM        = rho_cM,
        eff_rank_emp  = eff_emp,
        eff_rank_exp  = eff_exp,
        ratio_emp_exp = eff_emp / eff_exp,       # empirical / expected
        B_rec_emp     = 5 * eff_emp,             # 5x rule from empirical rank
        row.names = NULL
      )
    }, error = function(e) {
      message("FAILED: ", basename(f), " : ", conditionMessage(e)); NULL
    })

    parts <- mclapply(files, process_one, mc.cores = ${numThreads})
    ok    <- !vapply(parts, is.null, logical(1))
    dat   <- do.call(rbind, parts[ok])

    saveRDS(dat, ${_output[0]:r})
    write.csv(dat, ${_output[1]:r}, row.names = FALSE)

    cat(sprintf("processed %d/%d files -> %d rows\n", sum(ok), length(files), nrow(dat)))
    print(summary(dat[, c("eff_rank_emp", "eff_rank_exp", "ratio_emp_exp")]))

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary
# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J effrank_summary
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=16000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/22.Supp.GTEx_UKB_Effective_Rank.ipynb" effrank \
    --res_dir WORK_DIR/res/ \
    --cwd WORK_DIR/summary
EOF
base_sh="base_script"
output_script="summary_effrank.sh"
cat ${base_sh} | sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary
# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J effrank_summary
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=16000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/22.Supp.GTEx_UKB_Effective_Rank.ipynb" effrank \
    --res_dir WORK_DIR/result/ \
    --cwd WORK_DIR/summary
EOF
base_sh="base_script"
output_script="summary_effrank.sh"
cat ${base_sh} | sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}